In [3]:
import cv2
import mediapipe as mp
import numpy as np

EAR_THRESHOLD = 0.21
CONSECUTIVE_FRAMES = 20
BLINK_COUNTER = 0

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5)
LEFT_EYE = [362, 385, 387, 263, 373, 380]

def calculate_ear(landmarks, eye_indices, img_w, img_h):
    pts = []
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append(np.array([lm.x * img_w, lm.y * img_h]))
    v1 = np.linalg.norm(pts[1] - pts[5])
    v2 = np.linalg.norm(pts[2] - pts[4])
    h = np.linalg.norm(pts[0] - pts[3])
    return (v1 + v2) / (2.0 * h)

cap = cv2.VideoCapture(0)
while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)
    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            ear = calculate_ear(face_landmarks.landmark, LEFT_EYE, w, h)
            cv2.putText(frame, f"EAR: {ear:.2f}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            if ear < EAR_THRESHOLD:
                BLINK_COUNTER += 1
                if BLINK_COUNTER >= CONSECUTIVE_FRAMES:
                    cv2.putText(frame, "!!! DROWSY DRIVER !!!", (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            else:
                BLINK_COUNTER = 0
    cv2.imshow('Driver Alert System', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break
cap.release()
cv2.destroyAllWindows()